# Clase 4 — Datos y Sensores del Dominio Pesquero
## 🟢 Nivel NOVATO — Primer contacto con los datos del mar

**Curso:** Inteligencia Artificial Aplicada a la Producción Pesquera
**Institución:** UTN FRCh · PesquerosEnIA · 2026
**Docentes:** Ariel Giamportone · Soraya Corvalán

---

### ¿Para quién es este notebook?

Para vos que **nunca programaste** y querés entender de qué hablamos cuando decimos
"datos" en la pesca. **No necesitás escribir código**: cada bloque gris ya está listo.
Solo tenés que **apretar ▶ (o Shift+Enter) y mirar el resultado**.

> La idea no es que aprendas a programar hoy, sino que **veas qué se puede hacer**
> con los datos que tu barco o tu planta ya generan.

**Vamos a ver 3 cosas:**
1. 🌡️ La **temperatura del mar** y por qué cambia según la zona
2. 🌿 La **clorofila** (la "comida" que atrae a los peces)
3. 🎣 A qué temperatura **se pesca más** merluza

## Paso 0 — Preparar las herramientas

El bloque de abajo carga las herramientas de Python que dibujan los gráficos.
**No hace falta que entiendas cada línea.** Apretá ▶ y esperá el ✓ verde.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo para gráficos
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
np.random.seed(42)  # Reproducibilidad

print('✓ Librerías cargadas correctamente')
print(f'  NumPy {np.__version__} | Pandas {pd.__version__}')

## 1. 🌡️ La temperatura del mar (SST)

**SST** = *Sea Surface Temperature* = la temperatura del agua en la superficie.

¿Por qué importa? Porque **cada especie prefiere una temperatura**:
- 🐟 **Merluza:** aguas frías-templadas (4 a 12 °C)
- 🦑 **Calamar:** un poco más cálidas (8 a 16 °C)
- 🦐 **Langostino:** 6 a 14 °C

Si sabés la temperatura del agua, tenés una **pista** de dónde puede estar el recurso.

El próximo bloque arma un mapa de temperatura del mar argentino. Apretá ▶.

In [ ]:
# ── Parámetros geográficos de la PCA ──────────────────────────────────────────
# Latitudes: de Tierra del Fuego (-55°S) a Buenos Aires (-34°S)
# Longitudes: plataforma continental (-65°O a -44°O)
latitudes = np.arange(-55, -34, 0.5)
longitudes = np.arange(-65, -44, 0.5)
lon_grid, lat_grid = np.meshgrid(longitudes, latitudes)

# ── SST media anual: gradiente latitudinal realista ───────────────────────────
# Norte (~-35°S): ~18°C | Sur (~-55°S): ~5°C
# Pendiente: ~0.65°C por grado de latitud
sst_base = 18 + 0.65 * lat_grid

# Variabilidad espacial: efecto de frente Malvinas-Brasil cerca del talud
ruido_espacial = np.random.normal(0, 0.8, lon_grid.shape)
gradiente_lon = 0.05 * (lon_grid + 52)  # frente cerca de la isobata de 200m
sst_media_anual = sst_base + ruido_espacial + gradiente_lon

print(f'Grid generado: {lat_grid.shape[0]} latitudes × {lon_grid.shape[1]} longitudes')
print(f'SST media: {sst_media_anual.mean():.1f}°C | Mín: {sst_media_anual.min():.1f}°C | Máx: {sst_media_anual.max():.1f}°C')

In [ ]:
# ── Mapa de SST media anual ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))

contour = ax.contourf(longitudes, latitudes, sst_media_anual,
                      levels=20, cmap='RdYlBu_r', alpha=0.9)
cbar = plt.colorbar(contour, ax=ax, shrink=0.8)
cbar.set_label('Temperatura (°C)', fontsize=12)

# Marcar puertos principales
puertos = {
    'Puerto Madryn': (-42.77, -65.03),
    'Mar del Plata': (-38.00, -57.53),
    'Rawson': (-43.30, -65.10),
    'Ushuaia': (-54.80, -68.30)
}
for nombre, (lat_p, lon_p) in puertos.items():
    ax.plot(lon_p, lat_p, 'k^', markersize=8)
    ax.annotate(nombre, (lon_p + 0.3, lat_p), fontsize=8, color='black')

# Línea aproximada del frente Malvinas-Brasil
ax.axhline(-40, linestyle='--', color='white', linewidth=1.5,
           label='Frente Malvinas-Brasil (~40°S)')

ax.set_xlabel('Longitud (°O)', fontsize=12)
ax.set_ylabel('Latitud (°S)', fontsize=12)
ax.set_title('Temperatura Superficial del Mar — Media Anual\nPlataforma Continental Argentina',
             fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

### 👀 ¿Qué se ve en el mapa?

- Los colores **rojos** (arriba, cerca de Buenos Aires) = agua **más cálida**.
- Los colores **azules** (abajo, cerca de Ushuaia) = agua **más fría**.
- La línea blanca punteada marca el **frente Malvinas-Brasil**: donde se junta el agua
  fría del sur con la cálida del norte. **Ahí suele haber mucha pesca.**

👉 *Regla simple: el sur es más frío que el norte. La merluza vive donde el agua está fresca.*

## 2. 🌿 La clorofila: la "comida" del mar

La **clorofila** mide cuánto **fitoplancton** hay (unas plantitas microscópicas).
Más fitoplancton → más comida → más peces pequeños → más merluza y calamar.

**Donde hay mucha clorofila, suele haber pesca.** El próximo bloque compara los dos mapas
(temperatura y clorofila) lado a lado. Apretá ▶.

In [ ]:
# ── Clorofila-a: distribución espacial en la PCA ──────────────────────────────
np.random.seed(42)

# Alta productividad en la zona del frente (~-40°S) y golfos patagónicos
clorofila = np.abs(
    2.0 * np.exp(-((lat_grid + 40) ** 2) / 20)  # pico en el frente ~-40°S
    + 1.5 * np.exp(-((lat_grid + 43) ** 2) / 15)  # golfo San Jorge
    + 0.8 * np.exp(-((lat_grid + 47) ** 2) / 10)  # golfo San Matías
    + np.random.exponential(0.3, lon_grid.shape)   # ruido realista
)

# Figura comparativa: SST vs Clorofila
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# SST
c1 = axes[0].contourf(longitudes, latitudes, sst_media_anual, levels=15, cmap='RdYlBu_r')
plt.colorbar(c1, ax=axes[0]).set_label('SST (°C)')
axes[0].set_title('Temperatura Superficial del Mar (SST)', fontweight='bold')
axes[0].set_xlabel('Longitud (°O)')
axes[0].set_ylabel('Latitud (°S)')

# Clorofila
c2 = axes[1].contourf(longitudes, latitudes, clorofila, levels=15, cmap='YlGn')
plt.colorbar(c2, ax=axes[1]).set_label('Clorofila-a (mg/m³)')
axes[1].set_title('Clorofila-a (productividad primaria)', fontweight='bold')
axes[1].set_xlabel('Longitud (°O)')
axes[1].set_ylabel('Latitud (°S)')

plt.suptitle('PCA: SST vs Clorofila-a — indicadores de zonas de pesca',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 👀 ¿Qué se ve?

- El mapa de la **derecha** (verde) muestra dónde hay más **clorofila** (comida).
- Fijate que las zonas verdes coinciden con los **golfos** y con el **frente**.
- Un buen pesquero cruza mentalmente estos dos mapas: **temperatura adecuada + mucha comida = buena zona.**

## 3. 🎣 ¿A qué temperatura se pesca más?

Ahora vamos a usar **datos de 500 mareas** (viajes de pesca) para ver la relación
entre la temperatura del agua y cuántas toneladas de merluza se capturaron.

El primer bloque arma esos datos; el segundo hace el gráfico. Apretá ▶ en los dos.

In [ ]:
# ── Dataset de capturas históricas (partes de pesca simulados) ─────────────────
np.random.seed(42)
n_mareas = 500

# Variables oceanográficas al momento de la marea
sst_marea = np.random.normal(10, 3.5, n_mareas)   # SST en zona de pesca
chl_marea = np.abs(np.random.exponential(1.8, n_mareas))  # Clorofila
profundidad_m = np.random.uniform(60, 280, n_mareas)
mes = np.random.randint(1, 13, n_mareas)
lat_captura = np.random.uniform(-50, -39, n_mareas)
lon_captura = np.random.uniform(-62, -50, n_mareas)

# Captura de merluza (tn) — función de SST con óptimo en 8-12°C
prob_exito_sst = np.exp(-((sst_marea - 10) ** 2) / 18)
prob_exito_chl = np.clip(chl_marea / 4, 0, 1)
prob_exito_prof = np.exp(-((profundidad_m - 140) ** 2) / 6000)

captura_base = 60 * (0.45 * prob_exito_sst + 0.3 * prob_exito_chl + 0.25 * prob_exito_prof)
captura_merluza_tn = np.abs(captura_base + np.random.normal(0, 10, n_mareas))

registros_captura = pd.DataFrame({
    'mes': mes,
    'lat_captura': lat_captura,
    'lon_captura': lon_captura,
    'profundidad_m': profundidad_m,
    'sst_grados': sst_marea,
    'clorofila_mg_m3': chl_marea,
    'captura_merluza_tn': captura_merluza_tn
})

# Agregar estación
registros_captura['estacion'] = pd.cut(
    registros_captura['mes'],
    bins=[0, 3, 6, 9, 12],
    labels=['Verano', 'Otoño', 'Invierno', 'Primavera'])

print(f'Partes de pesca generados: {len(registros_captura)}')
print(f'Captura promedio por marea: {registros_captura["captura_merluza_tn"].mean():.1f} tn')
registros_captura.describe().round(2)

In [ ]:
# Captura promedio según la temperatura del agua
rangos_temp = pd.cut(registros_captura['sst_grados'], bins=[0, 6, 9, 12, 15, 20])
captura_promedio = registros_captura.groupby(rangos_temp, observed=True)['captura_merluza_tn'].mean()

plt.figure(figsize=(9, 5))
captura_promedio.plot(kind='bar', color='steelblue', edgecolor='white')
plt.xlabel('Temperatura del agua (°C)')
plt.ylabel('Captura promedio (toneladas)')
plt.title('¿A qué temperatura se pesca más merluza?')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print('👉 La captura es mayor cuando el agua está templada, entre 9 y 12 °C aprox.')


### 👀 La conclusión para tu barco

Las barras más altas están en el rango **9–12 °C**: ahí se captura más merluza.
Con un dato **gratis** (la temperatura del mar por satélite), podés **priorizar zonas**
antes de salir, ahorrando combustible y tiempo.

> Esto es, en pocas palabras, de lo que trata la IA aplicada a la pesca:
> **usar datos que ya existen para tomar mejores decisiones.**

## 4. 🆓 ¿De dónde saco estos datos? (todos gratis)

No hace falta comprar nada. Estas plataformas dan datos abiertos del mar argentino:

| Plataforma | Qué te da | Para qué sirve |
|-----------|-----------|----------------|
| **Copernicus Marine** | Temperatura y clorofila por satélite | Elegir zona antes de salir |
| **NOAA** | Temperatura del mar de alta resolución | Lo mismo, otra fuente |
| **Global Fishing Watch** | Dónde están pescando los barcos | Ver actividad de la flota |
| **INIDEP** | Capturas y estado de los recursos (Argentina) | Datos oficiales del país |

*No hace falta que las uses hoy. Solo tené presente que la información existe y es accesible.*

## ✅ Síntesis — Qué te llevás de esta clase

1. Tu barco y tu planta **ya generan datos** (temperatura, posición, capturas...).
2. La **temperatura del mar** y la **clorofila** son pistas de dónde hay recurso.
3. La merluza se pesca mejor en agua **templada (9–12 °C)**.
4. Todos estos datos están **disponibles gratis** en plataformas abiertas.

### Para seguir (cuando quieras)
- Volvé a correr los bloques y **cambiá algún número** para ver qué pasa (no se rompe nada).
- El **Nivel Intermedio** de esta misma clase te enseña a leer y modificar el código.
- Comunidad **PesquerosEnIA:** https://github.com/PesquerosEnIA

*¡Felicitaciones! Corriste tu primer análisis de datos pesqueros. 🎣*